In [12]:
import os
import json
import glob
import pymupdf
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

load_dotenv()

DATA_DIR = "../../data/raw"
OUTPUT_FILE = "../../data/processed/steward_decisions.json"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def extract_fields(text: str) -> dict:
    prompt = f"""다음 FIA 스튜어드 결정문에서 아래 필드만 추출해서 다른 텍스트 없이JSON으로만 반환해주세요.

필드:
- grand_prix: 그랑프리 이름
- year: 연도
- fact: 사실 관계
- infringement: 위반 조항
- decision: 결정
- reason: 판단 근거

결정문:
{text}

JSON:"""

    response = llm.invoke(prompt).content
    response = response.replace("```json", "").replace("```", "").strip()
    return json.loads(response)

pdf_files = glob.glob(f"{DATA_DIR}/**/*.pdf", recursive=True)
pdf_files = [f for f in pdf_files if any(gp in f for gp in ["australia_f1", "chinese_f1", "japnanese_f1"])]

results = []

for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path)
    print(f"처리 중: {filename}")

    with pymupdf.open(pdf_path) as pdf:
        text = "\n".join(page.get_text() for page in pdf)

    try:
        fields = extract_fields(text)
        fields["source"] = filename
        results.append(fields)
        print(f"완료: {filename}")
    except Exception as e:
        print(f"실패: {filename} - {e}")

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"총 {len(results)}개 저장 {OUTPUT_FILE}")

처리 중: 2026 Australian Grand Prix - Decision - Car 10 - Alleged causing a collision with Car 31.pdf
완료: 2026 Australian Grand Prix - Decision - Car 10 - Alleged causing a collision with Car 31.pdf
처리 중: 2026 Australian Grand Prix - Decision - Car 11 - Alleged forcing another driver off the track.pdf
완료: 2026 Australian Grand Prix - Decision - Car 11 - Alleged forcing another driver off the track.pdf
처리 중: 2026 Australian Grand Prix - Decision - Car 12 - Alleged pitlane incident.pdf
완료: 2026 Australian Grand Prix - Decision - Car 12 - Alleged pitlane incident.pdf
처리 중: 2026 Australian Grand Prix - Decision - Car 18 - Failing to set a lap time within 107%.pdf
완료: 2026 Australian Grand Prix - Decision - Car 18 - Failing to set a lap time within 107%.pdf
처리 중: 2026 Australian Grand Prix - Decision - Car 27 - Start Procedure Infringement.pdf
완료: 2026 Australian Grand Prix - Decision - Car 27 - Start Procedure Infringement.pdf
처리 중: 2026 Australian Grand Prix - Decision - Car 3 - Failing to s